# Data-Driven Multi-Touch Attribution

Pipeline outputs for the Markov and logistic-regression attribution models.

**Notebook goals**
- Inspect precomputed removal effects and regression odds without rerunning heavy jobs.
- Highlight the dominant customer journeys by frequency and value.
- Capture decision guidance bullets for marketing leadership.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
FIG_DIR = Path("../reports/figures")

markov_metrics = pd.read_csv(DATA_DIR / "attribution_markov_channel_metrics.csv")
removal_effects = pd.read_csv(DATA_DIR / "markov_removal_effects.csv")
regression_metrics = pd.read_csv(DATA_DIR / "attribution_regression_channel_metrics.csv")
heuristic_channels = pd.read_csv(DATA_DIR / "attribution_channel_metrics.csv")
regression_odds = pd.read_csv(DATA_DIR / "regression_channel_odds.csv")
touch_level = pd.read_csv(DATA_DIR / "attribution_touch_level.csv")

total_revenue = float(markov_metrics["attributed_revenue"].sum())

## Markov removal effects
Largest incremental revenue losses if a channel is removed.

In [ ]:
top_removal = removal_effects.nlargest(5, "removal_value").copy()
if "removal_share" not in top_removal.columns:
    top_removal["removal_share"] = top_removal["removal_value"] / total_revenue
top_removal["removal_share_pct"] = top_removal["removal_share"] * 100
top_removal_display = top_removal[["channel", "removal_value", "removal_share_pct"]].rename(
    columns={"removal_value": "removal_value_usd", "removal_share_pct": "removal_share_%"}
)
top_removal_display

## Journeys and top paths
First-touch view of the most common and most valuable channel sequences.

In [ ]:
base_touches = touch_level[touch_level["model_name"] == "first_touch"].copy()
paths = (
    base_touches.sort_values(["conversion_id", "touch_order"]).groupby("conversion_id")
    .agg(path=("channel", lambda x: " > ".join(x)), conversion_value=("conversion_value", "first"))
    .reset_index()
)
path_summary = (
    paths.groupby("path")
    .agg(conversions=("conversion_id", "count"), revenue=("conversion_value", "sum"))
    .reset_index()
    .sort_values("conversions", ascending=False)
)
top_path_frequency = path_summary.head(10).copy()
top_path_revenue = path_summary.sort_values("revenue", ascending=False).head(10).copy()
display(top_path_frequency)
display(top_path_revenue)

## Regression interpretability
Channel-level odds ratios plus validation AUC for the logistic model.

In [ ]:
auc_series = regression_odds["model_auc"].dropna()
auc = float(auc_series.iloc[0]) if not auc_series.empty else float("nan")
channel_odds = regression_odds[regression_odds["feature"].str.startswith("touch_count_")].copy()
channel_odds["channel"] = channel_odds["feature"].str.replace("touch_count_", "", regex=False)
top_lift = channel_odds.sort_values("odds_ratio", ascending=False).head(10)[["channel", "odds_ratio", "coefficient"]]
top_suppressed = channel_odds.sort_values("odds_ratio", ascending=True).head(10)[["channel", "odds_ratio", "coefficient"]]
print(f"Validation AUC: {auc:.3f}")
display(top_lift)
print("Channels associated with the lowest odds ratios")
display(top_suppressed)

## Decision guidance
Data-backed talking points for budget conversations.

In [ ]:
channel_roi = heuristic_channels.groupby("channel")["ROI"].mean()
lagging_channel = channel_roi.idxmin()
lagging_roi = channel_roi.loc[lagging_channel]
markov_star = top_removal.iloc[0]
regression_star = regression_metrics.sort_values("attributed_revenue", ascending=False).iloc[0]
path_star = top_path_frequency.iloc[0]
path_share = path_star["conversions"] / path_summary["conversions"].sum()
decision_md = f"""
- Removing **{markov_star['channel']}** risks ${markov_star['removal_value']:,.0f} ({markov_star['removal_share'] * 100:.1f}% of attributed revenue).
- **{regression_star['channel']}** tops the regression run with ROI {regression_star['ROI']:.1f}x; keep incremental dollars flowing there.
- The journey **{path_star['path']}** now accounts for {path_share:.1%} of logged conversions; sync messaging between those touches.
- **{lagging_channel}** remains ROI-light (~{lagging_roi:.1f}x); fence spend for experiments before scaling.
"""
display(Markdown(decision_md))